In [30]:
import os
import random
import pandas as pd
import unicodedata
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity

dataset_path = 'qa_dataset.csv'

SIMILARITY_THRESHOLD = 0.65 
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

In [31]:
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    # remover acentos
    nfkd = unicodedata.normalize('NFD', text)
    no_accents = ''.join([c for c in nfkd if unicodedata.category(c) != 'Mn'])
    # minúsculas e colapso de espaços
    s = re.sub(r'\s+', ' ', no_accents).strip().lower()
    return s


In [40]:
if not os.path.exists(dataset_path):
    perguntas = [
        "Qual é o jogo mais popular atualmente?",
        "Qual a melhor motocicleta para iniciantes?",
        "Quais jogos têm o melhor modo multiplayer?",
        "Qual moto tem o melhor desempenho em pistas?",
        "Qual é a moto mais veloz do mundo?",
        "Qual jogo tem a melhor história?",
        "Qual moto o Rogerio tem?"
    ]
    respostas = [
        "O jogo mais popular atualmente é League of Legends.",
        "A melhor motocicleta para iniciantes é o patinete elétrico.",
        "Os jogos com o melhor modo multiplayer são [Jogo A] e [Jogo B].",
        "A moto com o melhor desempenho em pistas é a [Moto Z].",
        "A moto mais veloz do mundo é a do Marc Marques.",
        "O jogo com a melhor história é Tomb Raider.",
        "Ele tem uma TDM 850 e uma NC 750x, ambas são bem bacanas."
    ]
    intents = [
        "informacao_jogo",
        "informacao_moto",
        "informacao_jogo",
        "informacao_moto",
        "informacao_moto",
        "informacao_jogo",
        "informacao_rogerio"
    ]
    df_init = pd.DataFrame({
        'question': perguntas,
        'answer': respostas,
        'intent': intents
    })
    df_init.to_csv(dataset_path, index=False, encoding='utf-8-sig')
    print("Dataset inicial criado em", dataset_path)
else:
    print("Dataset encontrado em", dataset_path)

# carregar dataset
df = pd.read_csv(dataset_path, encoding='utf-8-sig')
# criar coluna normalized_question para facilitar buscas
df['normalized_question'] = df['question'].apply(normalize_text)
df.head(6)


Dataset encontrado em qa_dataset.csv


,question,answer,intent,normalized_question
0,Qual é o jogo mais popular atualmente?,O jogo mais popular atualmente é League of Leg...,informacao_jogo,qual e o jogo mais popular atualmente?
1,Qual a melhor motocicleta para iniciantes?,A melhor motocicleta para iniciantes é o patin...,informacao_moto,qual a melhor motocicleta para iniciantes?
2,Quais jogos têm o melhor modo multiplayer?,Os jogos com o melhor modo multiplayer são [Jo...,informacao_jogo,quais jogos tem o melhor modo multiplayer?
3,Qual moto tem o melhor desempenho em pistas?,A moto com o melhor desempenho em pistas é a [...,informacao_moto,qual moto tem o melhor desempenho em pistas?
4,Qual é a moto mais veloz do mundo?,A moto mais veloz do mundo é a do Marc Marques.,informacao_moto,qual e a moto mais veloz do mundo?
5,Qual jogo tem a melhor história?,O jogo com a melhor história é Tomb Raider.,informacao_jogo,qual jogo tem a melhor historia?


In [33]:
def save_dataset(df: pd.DataFrame, path: str = dataset_path):
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print("Dataset salvo em", path)


In [ ]:
def train_model(df: pd.DataFrame):

    le = LabelEncoder()
    intents = df['intent'].fillna('unknown').astype(str).values
    labels = le.fit_transform(intents)

    vect = TfidfVectorizer()
    clf = SGDClassifier(random_state=RANDOM_SEED)
    X = vect.fit_transform(df['normalized_question'].values)
    clf.fit(X, labels)

    tfidf_matrix = X
    return {
        'vectorizer': vect,
        'classifier': clf,
        'label_encoder': le,
        'tfidf_matrix': tfidf_matrix
    }


model_bundle = train_model(df)
print("Treinamento completo. Intents:", list(model_bundle['label_encoder'].classes_))


Treinamento completo. Intents: ['informacao_jogo', 'informacao_moto', 'informacao_rogerio']


In [36]:
def find_similar_question(model_bundle, df: pd.DataFrame, user_question: str, top_k: int = 3):
    vect = model_bundle['vectorizer']
    matrix = model_bundle['tfidf_matrix']
    q_norm = normalize_text(user_question)
    q_vec = vect.transform([q_norm])
    sims = cosine_similarity(q_vec, matrix).flatten()
    # ordena índices por similaridade
    idx_sorted = sims.argsort()[::-1]
    results = []
    for idx in idx_sorted[:top_k]:
        results.append((idx, sims[idx], df.iloc[idx]['question'], df.iloc[idx]['answer'], df.iloc[idx]['intent']))
    return results


In [38]:
def responder_interativo(user_question: str, df: pd.DataFrame, model_bundle, ask_when_unknown: bool = True):
    # 1) similaridade
    sims = find_similar_question(model_bundle, df, user_question, top_k=1)
    if sims:
        idx, score, q_text, a_text, intent = sims[0]
        if score >= SIMILARITY_THRESHOLD:
            return {
                'answer': a_text,
                'reason': f'similarity ({score:.2f}) with stored question: "{q_text}"',
                'learned': False
            }

    # 2) prever intenção
    q_norm = normalize_text(user_question)
    vect = model_bundle['vectorizer']
    clf = model_bundle['classifier']
    le = model_bundle['label_encoder']
    q_vec = vect.transform([q_norm])
    pred_label = clf.predict(q_vec)[0]
    pred_intent = le.inverse_transform([pred_label])[0]

    # 3) buscar respostas existentes para a intenção
    candidates = df[df['intent'] == pred_intent]
    if not candidates.empty:
        # escolher resposta aleatória entre as existentes para essa intent
        chosen = candidates.sample(1, random_state=RANDOM_SEED).iloc[0]
        return {
            'answer': chosen['answer'],
            'reason': f'predicted intent "{pred_intent}"',
            'learned': False
        }

    # 4) se chegou aqui, não temos resposta suficiente — perguntar ao usuário e salvar
    if ask_when_unknown:
        print("Não sei a resposta para essa pergunta. Me ensine, por favor.")
        given_answer = input("→ Escreva a resposta adequada: ").strip()
        suggested_intent = input(f"→ Deseja definir uma etiqueta de intenção para essa pergunta? (enter para usar '{pred_intent}' ou escreva uma etiqueta): ").strip()
        if suggested_intent == '':
            suggested_intent = pred_intent if pred_intent else 'unknown'
        # adicionar ao dataframe
        new_row = {
            'question': user_question,
            'answer': given_answer,
            'intent': suggested_intent,
            'normalized_question': normalize_text(user_question)
        }
        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
        save_dataset(df, dataset_path)
        # re-treinar
        new_bundle = train_model(df)
        print("Obrigado. Aprendi essa pergunta e atualizei o modelo.")
        return {
            'answer': given_answer,
            'reason': 'learned_from_user',
            'learned': True,
            'model_bundle': new_bundle,
            'df': df
        }
    else:
        return {
            'answer': None,
            'reason': 'unknown',
            'learned': False
        }
